# AI-Assisted De Novo Lead Generation for Oral GLP-1 Receptor Agonists

This notebook documents a medicinal-chemistry inspired, computational lead-generation workflow for a non-peptide GLP-1 receptor agonist program.

It captures:
- the therapeutic motivation,
- the target selection rationale,
- the scaffold design strategy,
- the generation of candidate SMILES,
- oral drug-likeness screening,
- and 3D conformer generation.

## Important note

This notebook does **not** claim biological validation, receptor agonism, or clinical efficacy.  
It documents an early-stage **lead-generation** workflow suitable for a portfolio or GitHub project.


## 1. Project Motivation

The GLP-1 receptor is a major therapeutic target in metabolic disease and weight management.  
Existing GLP-1 therapies are largely peptide-based and often require injection.

The goal of this project was to explore whether a **non-peptide, orally compatible small molecule scaffold** could be designed using simple medicinal chemistry rules and then screened for basic drug-likeness.


## 2. Target Definition

### Target
Human GLP-1 receptor (GLP1R)

### Why this target?
- Strong commercial relevance in obesity and type 2 diabetes
- High unmet need for oral alternatives
- Large market opportunity for small-molecule agonists
- Potential advantages in manufacturing and patient convenience


In [ ]:
# Cell 1 — Install dependencies
!pip install -q rdkit requests pandas


In [ ]:
# Cell 2 — Imports
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, AllChem


## 3. Scaffold Design Strategy

A benzimidazole–pyridine-inspired core was selected as the starting point.

The design rationale used in this project was:
- maintain a compact, drug-like molecular weight,
- preserve aromatic and heteroaromatic features for target engagement potential,
- introduce a carboxylic acid to tune polarity and binding interactions,
- use fluorine substitution to improve metabolic stability,
- keep the scaffold synthetically plausible.

### Base scaffold
This is the initial template used for analog generation.


In [ ]:
# Cell 3 — Base scaffold
core_template_smiles = "O=C(O)c1ccc2nc(CN3CCC(c4cccc(OCC)n4)CC3)n(CC3CCO3)c2c1"
core_mol = Chem.MolFromSmiles(core_template_smiles)

print("Base scaffold valid:", core_mol is not None)
print(core_template_smiles)


## 4. Candidate Generation by Simple Medicinal Chemistry Modifications

The following analogs were generated by applying small structural edits to the base scaffold.

### Modification logic
- **Fluorination**: potential metabolic stabilization
- **Nitrile substitution**: polarity and electronic tuning
- **Expanded lipophilicity variant**: exploratory optimization of membrane permeability

These are not proof of activity. They are only lead-generation hypotheses.


In [ ]:
# Cell 4 — Generated analogs
modifications = {
    "Lead_Base_Unsubstituted": core_template_smiles,
    "Lead_Mod_Fluorinated": "O=C(O)c1ccc2nc(CN3CCC(c4cc(F)c(OCC)n4)CC3)n(CC3CCO3)c2c1",
    "Lead_Mod_Nitrile_Core": "N#Cc1ccc2nc(CN3CCC(c4cccc(OCC)n4)CC3)n(CC3CCO3)c2c1",
    "Lead_Mod_Oxetane_Expanded": "O=C(O)c1ccc2nc(CN3CCC(c4cccc(OCC(F)(F)F)n4)CC3)n(CC3CCO3)c2c1"
}

pd.DataFrame(
    [(k, v) for k, v in modifications.items()],
    columns=["Compound ID", "SMILES"]
)


## 5. Sanity Check: Structural Validity

Each generated SMILES is parsed with RDKit to confirm that the molecule is chemically valid before any downstream property calculations.


In [ ]:
# Cell 5 — Validate all molecules
validation_rows = []

for compound_id, smiles in modifications.items():
    mol = Chem.MolFromSmiles(smiles)
    validation_rows.append({
        "Compound ID": compound_id,
        "Valid": mol is not None
    })

validation_df = pd.DataFrame(validation_rows)
validation_df


## 6. Physicochemical Screening

The next step is to estimate whether the generated molecules fall within common oral drug-like boundaries using basic property filters.

### Filters used
- Molecular weight ≤ 500 Da
- LogP ≤ 5
- H-bond donors ≤ 5
- H-bond acceptors ≤ 10
- Rotatable bonds ≤ 10


In [ ]:
# Cell 6 — Compute oral drug-likeness metrics
results_data = []

for compound_id, smiles in modifications.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        hbd = Lipinski.NumHDonors(mol)
        hba = Lipinski.NumHAcceptors(mol)
        rot_bonds = Lipinski.NumRotatableBonds(mol)

        is_lipinski_compliant = (mw <= 500) and (logp <= 5.0) and (hbd <= 5) and (hba <= 10)
        is_veber_compliant = (rot_bonds <= 10)

        status = "PASSED (Highly Bioavailable)" if (is_lipinski_compliant and is_veber_compliant) else "FLAGGED (Requires Formulation Optimization)"

        results_data.append({
            "Compound ID": compound_id,
            "MolWt (Da)": round(mw, 2),
            "LogP": round(logp, 2),
            "H-Donors": hbd,
            "H-Acceptors": hba,
            "RotBonds": rot_bonds,
            "Oral Screen Status": status
        })

df_screening = pd.DataFrame(results_data)
df_screening


## 7. Lead Selection

The fluorinated analog was selected as the preferred lead candidate for further structural exploration because it retained oral-like physicochemical properties while introducing a stability-oriented substitution.


In [ ]:
# Cell 7 — Select the preferred lead
top_lead_smiles = modifications["Lead_Mod_Fluorinated"]

print("Selected lead SMILES:")
print(top_lead_smiles)


## 8. Corrected Lead Structure and 3D Preparation

A SMILES correction step was required to ensure RDKit could sanitize and kekulize the molecule properly.

The corrected SMILES used for 3D generation was:

`O=C(O)c1ccc2nc(CN3CCC(c4ccc(F)c(OCC)n4)CC3)n(CC3CCO3)c2c1`


In [ ]:
# Cell 8 — Load corrected lead and generate a 3D conformer
corrected_glp1_smiles = "O=C(O)c1ccc2nc(CN3CCC(c4ccc(F)c(OCC)n4)CC3)n(CC3CCO3)c2c1"

glp1_mol = Chem.MolFromSmiles(corrected_glp1_smiles)

print("Molecule parsed successfully:", glp1_mol is not None)

if glp1_mol:
    glp1_mol_h = Chem.AddHs(glp1_mol)

    # Generate 3D coordinates
    status = AllChem.EmbedMolecule(glp1_mol_h, AllChem.ETKDGv3())

    # Optimize geometry
    if status == 0:
        AllChem.UFFOptimizeMolecule(glp1_mol_h, maxIters=1000)

    mw = Descriptors.MolWt(glp1_mol)
    logp = Descriptors.MolLogP(glp1_mol)

    print("3D conformer generation completed.")
    print(f"Molecular Weight: {mw:.2f} Da")
    print(f"LogP: {logp:.2f}")


## 9. Project Interpretation

### What this notebook demonstrates
- A rational target-driven starting point
- De novo generation of candidate non-peptide GLP-1 receptor-like scaffolds
- Basic medicinal chemistry filtering
- A corrected lead structure suitable for follow-up work
- Successful 3D conformer generation and minimization

### What this notebook does not demonstrate
- Target binding
- Receptor agonism
- Docking score validation
- ADMET prediction
- Experimental efficacy

### Portfolio value
This notebook shows how an early-stage idea can be translated into a structured computational workflow with explicit design logic and reproducible screening.


In [ ]:
# Cell 10 — Final summary table
summary = pd.DataFrame([
    ["Therapeutic area", "Metabolic disease / weight management"],
    ["Target", "Human GLP-1 receptor (GLP1R)"],
    ["Design approach", "De novo non-peptide scaffold generation"],
    ["Preferred lead", "Lead_Mod_Fluorinated"],
    ["Lead SMILES", top_lead_smiles],
    ["Corrected lead SMILES", corrected_glp1_smiles],
    ["Lead molecular weight", f"{Descriptors.MolWt(glp1_mol):.2f} Da"],
    ["Lead LogP", f"{Descriptors.MolLogP(glp1_mol):.2f}"],
], columns=["Metric", "Value"])

summary


## 10. Next Steps

The natural next steps after this notebook are:
1. Retrieve the human GLP1R sequence and structure metadata.
2. Prepare the receptor for docking.
3. Run structure-based docking against the corrected lead.
4. Compare scores against known ligands.
5. Expand the analog series and rank candidates by activity and drug-likeness.
6. Add ADMET and QSAR modeling.
